# Olist E-Commerce BI — Week 4: Power BI Executive Report

**Author:** Oluwaseyi ([@shykeys-data](https://github.com/shykeys-data)) · **Builds on:** Weeks 1–3
**Tools:** Python (data prep) · Power BI Desktop (model, DAX, report)

---

### What this notebook does
Python does the heavy lifting (cleaning, modelling, customer segments). Power BI does the business reporting.
This notebook exports the **same quality-checked star schema** used in Weeks 1–3 as CSV files that Power BI loads directly:

| File | Grain | Used for |
|---|---|---|
| `fact_orders.csv` | 1 row per order | Revenue, orders, late %, reviews |
| `fact_order_items.csv` | 1 row per item | Category and seller revenue |
| `dim_customer.csv` | 1 row per real customer | Repeat rate, RFM segment |
| `dim_date.csv` | 1 row per day | Date slicers and time intelligence |
| `dim_segment.csv` | 1 row per RFM segment | Suggested action per segment |
| `dim_timing.csv` | 1 row per delivery-timing band | Sort order for the timing chart |
| `retention_cohort.csv` | cohort × month | Retention heatmap |

> **How to run:** **Runtime → Run all**. The last cell downloads `olist_powerbi.zip`. Unzip it and follow `powerbi/BUILD_GUIDE.md` in the repo.

## 1. Download the data and load the shared data layer

In [ ]:
!pip install -q -U kagglehub

import os, sys, urllib.request, zipfile
import numpy as np
import pandas as pd

# The dashboard's data layer holds the quality-checked definitions (Weeks 1-3).
# Reusing it means Power BI gets exactly the same numbers as the notebooks and the live dashboard.
if not os.path.exists("data.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/shykeys-data/olist-ecommerce-bi/main/dashboard/data.py", "data.py")
sys.path.insert(0, ".")
import data

In [ ]:
DATA_DIR = os.environ.get("OLIST_DATA_DIR")
if DATA_DIR is None:
    from google.colab import userdata
    def get_secret(name):
        try:
            return userdata.get(name)
        except Exception:
            return None
    data.configure_kaggle({k: get_secret(k) for k in ["KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"] if get_secret(k)})
    DATA_DIR = data.download()
print("Data folder:", DATA_DIR)

model = data.build_model(DATA_DIR)
fo, foi = model["orders"], model["items"]
print(f"{len(fo):,} orders · {len(foi):,} items")

## 2. Build the Power BI tables

In [ ]:
# ---- fact_orders
fact_orders = pd.DataFrame({
    "order_id": fo["order_id"],
    "customer_unique_id": fo["customer_unique_id"],
    "order_status": fo["order_status"],
    "purchase_date": fo["order_purchase_timestamp"].dt.date,
    "customer_state": fo["state"],
    "main_category": fo["main_category"].fillna("Unknown"),
    "n_items": fo["n_items"],
    "order_value": fo["order_value"].round(2),
    "delivery_days": fo["delivery_days"].round(2),
    "days_late": fo["days_late"],
    "is_late": fo["is_late"],                       # 1 / 0, blank if not delivered
    "review_score": fo["review_score"],
    "review_after_delivery": np.where(fo["review_score"].notna() & fo["order_delivered_customer_date"].notna(),
                                      fo["review_after_delivery"].astype(float), np.nan),
})
fact_orders["delivery_timing"] = pd.cut(fact_orders["days_late"], bins=data.TIMING_BINS,
                                        labels=data.TIMING_LABELS).astype(object).fillna("")

# ---- fact_order_items
fact_order_items = pd.DataFrame({
    "order_id": foi["order_id"], "order_item_id": foi["order_item_id"],
    "product_id": foi["product_id"], "seller_id": foi["seller_id"],
    "category": foi["category"].fillna("Unknown"),
    "price": foi["price"], "freight_value": foi["freight_value"], "item_total": foi["item_total"].round(2),
})

# ---- dim_customer with RFM segment (purchase-DAY definition from the Week 2.1 quality check)
p = fo[~fo["order_status"].isin(["canceled", "unavailable"])].copy()
p["order_value"] = p["order_value"].fillna(0)
p["day"] = p["order_purchase_timestamp"].dt.normalize()
snap = p["day"].max() + pd.Timedelta(days=1)
rfm = p.groupby("customer_unique_id").agg(customer_state=("state", "first"), first_purchase=("day", "min"),
                                         last_purchase=("day", "max"), purchase_days=("day", "nunique"),
                                         monetary=("order_value", "sum"))
rfm["recency_days"] = (snap - rfm["last_purchase"]).dt.days
rfm["R"] = pd.qcut(rfm["recency_days"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["M"] = pd.qcut(rfm["monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
F = rfm["purchase_days"].clip(upper=3)
cond = [(F >= 2) & (rfm["R"] >= 4), F >= 2, (F == 1) & (rfm["R"] >= 4) & (rfm["M"] >= 4),
        (F == 1) & (rfm["R"] >= 4), (F == 1) & (rfm["R"] <= 2) & (rfm["M"] >= 4), (F == 1) & (rfm["R"] <= 2)]
names = ["Champions", "Loyal (lapsing)", "New big spenders", "New customers", "Big spenders at risk", "Hibernating"]
rfm["segment"] = np.select(cond, names, default="Needs attention")
rfm["is_repeat"] = (rfm["purchase_days"] > 1).astype(int)
rfm["cohort_month"] = rfm["first_purchase"].dt.to_period("M").dt.to_timestamp().dt.date
rfm["first_purchase"] = rfm["first_purchase"].dt.date
rfm["last_purchase"] = rfm["last_purchase"].dt.date
rfm["monetary"] = rfm["monetary"].round(2)
dim_customer = rfm.reset_index()[["customer_unique_id", "customer_state", "first_purchase", "last_purchase",
                                  "cohort_month", "purchase_days", "is_repeat", "monetary", "recency_days",
                                  "R", "M", "segment"]]

# ---- small lookup dimensions
dim_segment = pd.DataFrame({
    "segment": ["Champions", "Loyal (lapsing)", "New big spenders", "New customers",
                "Needs attention", "Big spenders at risk", "Hibernating"],
    "segment_order": range(1, 8),
    "suggested_action": ["VIP perks, referral programme", "Personal win-back offer",
                         "Fast 2nd-purchase nudge, cross-sell", "Onboarding, first-repeat discount",
                         "Reminder with recommendations", "Targeted win-back on their category",
                         "Low-cost reactivation only"]})
dim_timing = pd.DataFrame({"delivery_timing": data.TIMING_LABELS, "timing_order": range(1, 9),
                           "timing_side": ["Early / on time"] * 4 + ["Late"] * 4})

# ---- dim_date: every day from the first to the last order
days = pd.date_range(fo["order_purchase_timestamp"].min().normalize(), fo["order_purchase_timestamp"].max().normalize())
dim_date = pd.DataFrame({"date": days.date, "year": days.year, "quarter": "Q" + days.quarter.astype(str),
                         "month_number": days.month, "month_name": days.strftime("%b"),
                         "year_month": days.strftime("%Y-%m"), "month_start": days.to_period("M").to_timestamp().date,
                         "weekday": days.strftime("%a"), "weekday_number": days.dayofweek + 1})

# ---- retention heatmap (same function as the dashboard)
cust = data.customer_tables(fo)
retention_cohort = (cust["retention"].drop(columns=[0]).rename_axis("cohort_month").reset_index()
                    .melt(id_vars="cohort_month", var_name="months_after", value_name="pct_returned"))
retention_cohort["pct_returned"] = retention_cohort["pct_returned"].round(2)

tables = {"fact_orders": fact_orders, "fact_order_items": fact_order_items, "dim_customer": dim_customer,
          "dim_date": dim_date, "dim_segment": dim_segment, "dim_timing": dim_timing,
          "retention_cohort": retention_cohort}
for k, v in tables.items():
    print(f"{k:18s} {len(v):>8,} rows × {v.shape[1]} cols")

## 3. Quality checks before export
Every check must pass. These are the same rules as Weeks 1–3, so the Power BI report should match the notebooks and dashboard.

In [ ]:
delivered = fact_orders[fact_orders["order_status"] == "delivered"]
checks = {
    "fact_orders: one row per order": fact_orders["order_id"].is_unique,
    "dim_customer: one row per customer": dim_customer["customer_unique_id"].is_unique,
    "dim_date: one row per day": pd.Series(dim_date["date"]).is_unique,
    "Every order date exists in dim_date": fact_orders["purchase_date"].isin(set(dim_date["date"])).all(),
    "Every item's order exists in fact_orders": fact_order_items["order_id"].isin(set(fact_orders["order_id"])).all(),
    "Every segment exists in dim_segment": dim_customer["segment"].isin(set(dim_segment["segment"])).all(),
    "Every timing band exists in dim_timing": delivered.loc[delivered["delivery_timing"] != "", "delivery_timing"].isin(set(dim_timing["delivery_timing"])).all(),
    "Item revenue = order revenue (to the cent)": np.isclose(fact_order_items["item_total"].sum(), fact_orders["order_value"].sum(), atol=1),
    "Customer count matches dashboard function": len(dim_customer) == cust["customers"],
    "Repeat rate matches dashboard function": np.isclose(100 * dim_customer["is_repeat"].mean(), cust["repeat_rate"]),
}
for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
assert all(checks.values()), "A check failed. Don't export."

print()
print("Headline numbers Power BI should show (all dates):")
print(f"  Revenue (delivered):  R$ {delivered['order_value'].sum() / 1e6:,.2f}M")
print(f"  Delivered orders:     {len(delivered):,}")
print(f"  Average order value:  R$ {delivered['order_value'].mean():,.2f}")
print(f"  Delivered late:       {100 * delivered['is_late'].mean():.1f}%")
print(f"  Average review:       {delivered['review_score'].mean():.2f}")
print(f"  Customers:            {len(dim_customer):,}")
print(f"  Repeat rate:          {100 * dim_customer['is_repeat'].mean():.1f}%")

## 4. Export and download

In [ ]:
os.makedirs("olist_powerbi", exist_ok=True)
for k, v in tables.items():
    v.to_csv(f"olist_powerbi/{k}.csv", index=False, encoding="utf-8")
with zipfile.ZipFile("olist_powerbi.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for k in tables:
        z.write(f"olist_powerbi/{k}.csv", f"{k}.csv")
print(f"olist_powerbi.zip: {os.path.getsize('olist_powerbi.zip') / 1e6:.1f} MB")

In [ ]:
from google.colab import files
files.download("olist_powerbi.zip")

## Next: build the report in Power BI Desktop
Follow [`powerbi/BUILD_GUIDE.md`](../powerbi/BUILD_GUIDE.md): load the CSVs, create the relationships, paste the DAX measures from `measures.dax`, apply the `olist_theme.json` theme, then build the 3 report pages.